# SaúdePOP — Pipeline de Machine Learning e Análise de Dados

**PIM VI — Projeto Integrado Multidisciplinar — UNIP EaD**

Este caderno implementa o pipeline completo de análise e modelagem preditiva para o sistema de prontuário eletrônico e fila inteligente da Clínica Saúde Popular.

**Problemas abordados:**
1. Previsão de falta (no-show) — Classificação binária
2. Estimativa de tempo de espera — Regressão

**Etapas do pipeline:**
1. Carga e geração do dataset
2. Pré-processamento (limpeza, codificação, normalização)
3. Análise exploratória de dados (gráficos e estatísticas)
4. Modelagem — Classificação (Regressão Logística vs Random Forest)
5. Modelagem — Regressão (Regressão Linear vs Gradient Boosting)
6. Ajuste de hiperparâmetros (Grid Search)
7. Avaliação e comparação dos modelos
8. Salvamento dos resultados

## 1. Importações e Configuração

In [ ]:
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor, RandomForestClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
%matplotlib inline

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

OUTPUT_DIR = "resultados"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Bibliotecas importadas com sucesso.")

## 2. Geração e Carga do Dataset

O dataset simula 2.000 registros de agendamentos de consultas da Clínica Saúde Popular, representando 6 meses de operação. Os dados foram gerados para reproduzir padrões realistas de clínicas populares no Brasil.

In [ ]:
def gerar_dataset(n=2000):
    """Gera dataset simulado de atendimentos de clínica popular."""
    np.random.seed(42)

    dias = ["seg", "ter", "qua", "qui", "sex", "sab"]
    tipos = ["clinica_geral", "pediatria", "ginecologia", "ortopedia"]
    sexos = ["M", "F"]

    dados = {
        "id_agendamento": range(1, n + 1),
        "dia_semana": np.random.choice(dias, n, p=[0.20, 0.18, 0.17, 0.17, 0.16, 0.12]),
        "hora_agendamento": np.random.choice(range(7, 19), n),
        "tipo_consulta": np.random.choice(tipos, n, p=[0.40, 0.25, 0.20, 0.15]),
        "idade_paciente": np.random.normal(45, 18, n).clip(1, 95).astype(int),
        "sexo": np.random.choice(sexos, n, p=[0.42, 0.58]),
        "distancia_km": np.random.exponential(5, n).clip(0.5, 30).round(1),
        "consultas_anteriores": np.random.poisson(3, n),
        "faltas_anteriores": np.random.poisson(0.8, n).clip(0, 8),
        "qtd_fila": np.random.poisson(6, n).clip(0, 20),
    }

    df = pd.DataFrame(dados)

    # Modelar probabilidade de falta com base em fatores reais
    prob_falta = 0.15
    prob_falta += 0.08 * (df["dia_semana"] == "seg")
    prob_falta += 0.05 * (df["hora_agendamento"] >= 16)
    prob_falta += 0.02 * (df["distancia_km"] > 10)
    prob_falta += 0.05 * (df["faltas_anteriores"] >= 2)
    prob_falta -= 0.03 * (df["consultas_anteriores"] >= 5)
    prob_falta = prob_falta.clip(0.05, 0.60)

    df["compareceu"] = (np.random.random(n) > prob_falta).astype(int)

    # Modelar tempo de espera
    tempo_base = 15.0
    tempo_base += df["qtd_fila"] * 2.5
    tempo_base += 5.0 * (df["dia_semana"] == "seg")
    tempo_base += 3.0 * (df["hora_agendamento"].between(10, 12))
    tempo_base += np.random.normal(0, 5, n)
    df["tempo_espera_min"] = tempo_base.clip(2, 60).round(1)

    return df


# Gerar ou carregar dataset
CSV_PATH = "dados_atendimentos.csv"
if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)
    print(f"Dataset carregado: {CSV_PATH} ({len(df)} registros)")
else:
    df = gerar_dataset()
    df.to_csv(CSV_PATH, index=False)
    print(f"Dataset gerado e salvo: {CSV_PATH} ({len(df)} registros)")

df.head(10)

In [ ]:
# Visão geral do dataset
print(f"Shape: {df.shape}")
print(f"\nTipos de dados:")
print(df.dtypes)
print(f"\nEstatísticas descritivas:")
df.describe()

## 3. Pré-processamento dos Dados

### 3.1 Limpeza
Remoção de duplicatas e valores ausentes.

In [ ]:
# Verificar valores ausentes
print("Valores ausentes por coluna:")
print(df.isnull().sum())
print(f"\nDuplicatas: {df.duplicated().sum()}")

# Limpeza
n_antes = len(df)
df = df.drop_duplicates()
df = df.dropna()
n_depois = len(df)
print(f"\nRegistros removidos: {n_antes - n_depois}")
print(f"Registros restantes: {n_depois}")

### 3.2 Codificação de Variáveis Categóricas

Aplicação de One-Hot Encoding para as variáveis `dia_semana`, `tipo_consulta` e `sexo`.

In [ ]:
# Codificação One-Hot
df_ml = pd.get_dummies(df, columns=["dia_semana", "tipo_consulta", "sexo"], drop_first=True)

print(f"Colunas após encoding: {len(df_ml.columns)}")
print(f"\nNovas colunas criadas:")
novas = [c for c in df_ml.columns if c not in df.columns]
for c in novas:
    print(f"  - {c}")

### 3.3 Separação de Features e Variáveis Alvo

In [ ]:
# Separar features e targets
colunas_excluir = ["id_agendamento", "compareceu", "tempo_espera_min"]
feature_cols = [c for c in df_ml.columns if c not in colunas_excluir]

X = df_ml[feature_cols]
y_class = df_ml["compareceu"]       # Variável alvo para classificação (no-show)
y_reg = df_ml["tempo_espera_min"]    # Variável alvo para regressão (tempo de espera)

print(f"Features: {len(feature_cols)} variáveis")
print(f"Target classificação: compareceu (0=falta, 1=compareceu)")
print(f"Target regressão: tempo_espera_min (minutos)")
print(f"\nFeatures utilizadas:")
for f in feature_cols:
    print(f"  - {f}")

## 4. Análise Exploratória de Dados

In [ ]:
# Estatísticas gerais
print(f"Total de registros: {len(df)}")
print(f"Taxa de comparecimento: {df['compareceu'].mean():.1%}")
print(f"Taxa de falta (no-show): {1 - df['compareceu'].mean():.1%}")
print(f"\nTempo médio de espera: {df['tempo_espera_min'].mean():.1f} min")
print(f"Mediana do tempo de espera: {df['tempo_espera_min'].median():.1f} min")
print(f"Máximo tempo de espera: {df['tempo_espera_min'].max():.1f} min")

In [ ]:
# Gráfico 1: Taxa de falta por dia da semana
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Análise Exploratória — SaúdePOP", fontsize=16, fontweight='bold')

ordem_dias = ["seg", "ter", "qua", "qui", "sex", "sab"]
taxa_dia = df.groupby("dia_semana")["compareceu"].apply(lambda x: 1 - x.mean())
taxa_dia_ordenada = taxa_dia.reindex(ordem_dias)
axes[0, 0].bar(taxa_dia_ordenada.index, taxa_dia_ordenada.values, color="coral")
axes[0, 0].set_title("Taxa de Falta por Dia da Semana")
axes[0, 0].set_ylabel("Taxa de Falta")
axes[0, 0].set_ylim(0, 0.35)

# Gráfico 2: Distribuição do tempo de espera
axes[0, 1].hist(df["tempo_espera_min"], bins=20, color="steelblue", edgecolor="white")
axes[0, 1].set_title("Distribuição do Tempo de Espera")
axes[0, 1].set_xlabel("Minutos")
axes[0, 1].set_ylabel("Frequência")

# Gráfico 3: Taxa de falta por horário
taxa_hora = df.groupby("hora_agendamento")["compareceu"].apply(lambda x: 1 - x.mean())
axes[1, 0].plot(taxa_hora.index, taxa_hora.values, marker="o", color="darkred")
axes[1, 0].set_title("Taxa de Falta por Horário")
axes[1, 0].set_xlabel("Hora")
axes[1, 0].set_ylabel("Taxa de Falta")

# Gráfico 4: Distância por comparecimento
axes[1, 1].boxplot(
    [df[df["compareceu"] == 1]["distancia_km"], df[df["compareceu"] == 0]["distancia_km"]],
    labels=["Compareceu", "Faltou"]
)
axes[1, 1].set_title("Distância por Comparecimento")
axes[1, 1].set_ylabel("Distância (km)")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "analise_exploratoria.png"), dpi=150)
plt.show()
print("Gráficos salvos em resultados/analise_exploratoria.png")

In [ ]:
# Correlações com a variável alvo
print("Correlações com 'compareceu':")
numericas = df.select_dtypes(include=[np.number])
corr = numericas.corr()["compareceu"].drop("compareceu").sort_values(key=abs, ascending=False)
for var, val in corr.items():
    print(f"  {var}: {val:.3f}")

print(f"\n--- Taxa de falta por dia da semana ---")
for dia in ordem_dias:
    if dia in taxa_dia.index:
        print(f"  {dia}: {taxa_dia[dia]:.1%}")

## 5. Modelagem — Previsão de No-Show (Classificação)

### 5.1 Divisão treino/teste e normalização

In [ ]:
# Divisão treino/teste (80/20) com estratificação
X_train, X_test, y_train, y_test = train_test_split(
    X, y_class, test_size=0.2, random_state=42, stratify=y_class
)

# Normalização (StandardScaler)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Treino: {len(X_train)} registros")
print(f"Teste: {len(X_test)} registros")
print(f"\nDistribuição no treino: {y_train.value_counts().to_dict()}")
print(f"Distribuição no teste: {y_test.value_counts().to_dict()}")

### 5.2 Modelo 1 — Regressão Logística

Escolhido pela **interpretabilidade** — permite identificar quais variáveis mais influenciam a falta. Adequado para problemas binários com variáveis independentes mistas.

In [ ]:
# Treinar Regressão Logística
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)
y_prob_lr = lr.predict_proba(X_test_scaled)[:, 1]

print("=== Regressão Logística ===")
print(f"Acurácia: {accuracy_score(y_test, y_pred_lr):.3f}")
print(f"AUC-ROC: {roc_auc_score(y_test, y_prob_lr):.3f}")
print(f"\n{classification_report(y_test, y_pred_lr, target_names=['Falta', 'Compareceu'])}")

### 5.3 Modelo 2 — Random Forest

Escolhido pela capacidade de capturar **relações não lineares** e interações entre variáveis, além de oferecer feature importance nativa.

In [ ]:
# Treinar Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

print("=== Random Forest ===")
print(f"Acurácia: {accuracy_score(y_test, y_pred_rf):.3f}")
print(f"AUC-ROC: {roc_auc_score(y_test, y_prob_rf):.3f}")
print(f"\n{classification_report(y_test, y_pred_rf, target_names=['Falta', 'Compareceu'])}")

In [ ]:
# Feature Importance — Random Forest
importances = pd.Series(rf.feature_importances_, index=feature_cols)
importances = importances.sort_values(ascending=False)

print("Feature Importance (Top 10):")
for feat, imp in importances.head(10).items():
    print(f"  {feat}: {imp:.3f}")

# Gráfico de feature importance
fig, ax = plt.subplots(figsize=(10, 6))
top_feat = importances.head(10)
ax.barh(top_feat.index[::-1], top_feat.values[::-1], color="forestgreen")
ax.set_title("Feature Importance — Random Forest (No-Show)")
ax.set_xlabel("Importância")
plt.tight_layout()
plt.show()

### 5.4 Ajuste de Hiperparâmetros — Grid Search

Grid Search com validação cruzada 5-fold para otimizar o Random Forest.

In [ ]:
# Grid Search
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [5, 10, 15],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1,
    verbose=0,
)
grid_search.fit(X_train, y_train)

print(f"Melhores parâmetros: {grid_search.best_params_}")
print(f"Melhor AUC-ROC (CV): {grid_search.best_score_:.3f}")

# Avaliar modelo otimizado
best_rf = grid_search.best_estimator_
y_pred_best = best_rf.predict(X_test)
y_prob_best = best_rf.predict_proba(X_test)[:, 1]

print(f"\n=== Random Forest Otimizado ===")
print(f"AUC-ROC (teste): {roc_auc_score(y_test, y_prob_best):.3f}")
print(f"F1-Score (teste): {f1_score(y_test, y_pred_best):.3f}")
print(f"Acurácia (teste): {accuracy_score(y_test, y_pred_best):.3f}")

In [ ]:
# Comparação antes/depois do ajuste e visualizações
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Matriz de Confusão
cm = confusion_matrix(y_test, y_pred_best)
im = axes[0].imshow(cm, cmap="Blues")
axes[0].set_title("Matriz de Confusão — RF Otimizado")
axes[0].set_xlabel("Predito")
axes[0].set_ylabel("Real")
axes[0].set_xticks([0, 1])
axes[0].set_yticks([0, 1])
axes[0].set_xticklabels(["Falta", "Compareceu"])
axes[0].set_yticklabels(["Falta", "Compareceu"])
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=14)
plt.colorbar(im, ax=axes[0])

# Curva ROC
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_best)
axes[1].plot(fpr_lr, tpr_lr, label=f"Reg. Logística (AUC={roc_auc_score(y_test, y_prob_lr):.2f})")
axes[1].plot(fpr_rf, tpr_rf, label=f"Random Forest (AUC={roc_auc_score(y_test, y_prob_best):.2f})")
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[1].set_title("Curva ROC — Comparação")
axes[1].set_xlabel("Taxa de Falso Positivo")
axes[1].set_ylabel("Taxa de Verdadeiro Positivo")
axes[1].legend()

# Comparação de métricas
metricas = ['Acurácia', 'F1-Score', 'AUC-ROC']
val_lr = [accuracy_score(y_test, y_pred_lr), f1_score(y_test, y_pred_lr), roc_auc_score(y_test, y_prob_lr)]
val_rf = [accuracy_score(y_test, y_pred_best), f1_score(y_test, y_pred_best), roc_auc_score(y_test, y_prob_best)]
x_pos = np.arange(len(metricas))
width = 0.35
axes[2].bar(x_pos - width/2, val_lr, width, label='Reg. Logística', color='steelblue')
axes[2].bar(x_pos + width/2, val_rf, width, label='Random Forest', color='forestgreen')
axes[2].set_xticks(x_pos)
axes[2].set_xticklabels(metricas)
axes[2].set_title("Comparação de Modelos — Classificação")
axes[2].set_ylim(0, 1)
axes[2].legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "modelo_noshow.png"), dpi=150)
plt.show()
print("Gráficos salvos em resultados/modelo_noshow.png")

## 6. Modelagem — Estimativa de Tempo de Espera (Regressão)

### 6.1 Modelo 1 — Regressão Linear

Baseline simples para estimativa de tempo contínuo.

### 6.2 Modelo 2 — Gradient Boosting Regressor

Captura relações complexas entre variáveis e o tempo de espera.

In [ ]:
# Divisão treino/teste para regressão
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)

scaler_r = StandardScaler()
X_train_r_scaled = scaler_r.fit_transform(X_train_r)
X_test_r_scaled = scaler_r.transform(X_test_r)

# Regressão Linear
lr_reg = LinearRegression()
lr_reg.fit(X_train_r_scaled, y_train_r)
y_pred_lr_r = lr_reg.predict(X_test_r_scaled)

rmse_lr = np.sqrt(mean_squared_error(y_test_r, y_pred_lr_r))
mae_lr = mean_absolute_error(y_test_r, y_pred_lr_r)
r2_lr = r2_score(y_test_r, y_pred_lr_r)

print("=== Regressão Linear ===")
print(f"RMSE: {rmse_lr:.1f} min")
print(f"MAE: {mae_lr:.1f} min")
print(f"R²: {r2_lr:.3f}")

# Gradient Boosting Regressor
gb = GradientBoostingRegressor(n_estimators=200, max_depth=5, random_state=42)
gb.fit(X_train_r, y_train_r)
y_pred_gb = gb.predict(X_test_r)

rmse_gb = np.sqrt(mean_squared_error(y_test_r, y_pred_gb))
mae_gb = mean_absolute_error(y_test_r, y_pred_gb)
r2_gb = r2_score(y_test_r, y_pred_gb)

print("\n=== Gradient Boosting Regressor ===")
print(f"RMSE: {rmse_gb:.1f} min")
print(f"MAE: {mae_gb:.1f} min")
print(f"R²: {r2_gb:.3f}")

In [ ]:
# Visualizações de regressão
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Real vs Predito
axes[0].scatter(y_test_r, y_pred_lr_r, alpha=0.4, s=10, label="Reg. Linear", color="blue")
axes[0].scatter(y_test_r, y_pred_gb, alpha=0.4, s=10, label="Gradient Boosting", color="red")
lims = [0, 65]
axes[0].plot(lims, lims, "k--", alpha=0.5)
axes[0].set_xlabel("Tempo Real (min)")
axes[0].set_ylabel("Tempo Predito (min)")
axes[0].set_title("Real vs Predito — Tempo de Espera")
axes[0].legend()

# Comparação RMSE/MAE
modelos = ["Reg. Linear", "Gradient Boosting"]
rmses = [rmse_lr, rmse_gb]
maes = [mae_lr, mae_gb]
x_pos = np.arange(len(modelos))
width = 0.35
axes[1].bar(x_pos - width / 2, rmses, width, label="RMSE", color="steelblue")
axes[1].bar(x_pos + width / 2, maes, width, label="MAE", color="coral")
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(modelos)
axes[1].set_ylabel("Minutos")
axes[1].set_title("Comparação de Modelos — Regressão")
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "modelo_tempo_espera.png"), dpi=150)
plt.show()
print("Gráficos salvos em resultados/modelo_tempo_espera.png")

## 7. Resumo dos Resultados

### Classificação (No-Show)

O **Random Forest** foi selecionado como modelo principal por apresentar métricas superiores em todas as dimensões. Após o ajuste de hiperparâmetros via Grid Search, obteve-se melhoria adicional.

### Regressão (Tempo de Espera)

O **Gradient Boosting** foi selecionado, com erro médio absoluto aceitável para comunicação no painel de fila da clínica.

In [ ]:
# Tabela resumo
print("=" * 60)
print("RESUMO DOS RESULTADOS")
print("=" * 60)

print("\n--- Classificação (No-Show) ---")
print(f"{'Métrica':<25} {'Reg. Logística':<18} {'Random Forest':<18}")
print("-" * 60)
print(f"{'Acurácia':<25} {accuracy_score(y_test, y_pred_lr):<18.3f} {accuracy_score(y_test, y_pred_best):<18.3f}")
print(f"{'F1-Score':<25} {f1_score(y_test, y_pred_lr):<18.3f} {f1_score(y_test, y_pred_best):<18.3f}")
print(f"{'AUC-ROC':<25} {roc_auc_score(y_test, y_prob_lr):<18.3f} {roc_auc_score(y_test, y_prob_best):<18.3f}")

print("\n--- Regressão (Tempo de Espera) ---")
print(f"{'Métrica':<25} {'Reg. Linear':<18} {'Gradient Boosting':<18}")
print("-" * 60)
print(f"{'RMSE (min)':<25} {rmse_lr:<18.1f} {rmse_gb:<18.1f}")
print(f"{'MAE (min)':<25} {mae_lr:<18.1f} {mae_gb:<18.1f}")
print(f"{'R²':<25} {r2_lr:<18.3f} {r2_gb:<18.3f}")

print("\n" + "=" * 60)
print("PIPELINE CONCLUÍDO COM SUCESSO")
print(f"Resultados salvos em: {OUTPUT_DIR}/")
print("=" * 60)